In [1]:
"""
Domain-adapt Layer 2 to phone images WITHOUT retraining RT-DETR.

The RT-DETR encoder is frozen (you only ever trained the GOA head on cached
features). So "adapting Layer 2 to phone" = re-cache the encoder's 768-d
features on the HAM+PAD mix, then re-run the SAME GOA head search on the mixed
features. Encoder weights never move. ~40 min, mostly the GOA swarm.

This file REPLACES cell G1 of your GOA notebook and adds a per-domain +
escalated eval. Cells G2, G3, G4 from that notebook run UNCHANGED on the
Xtr/ytr/Xva/yva/Xte/yte this produces. Run P0-P3, then G2-G4, then P5.

Layer 2 is image-only — no metadata here (that lives in Layer 1).
Colab, GPU. Set CKPT_L2 to layer2_rtdetr.pt (the encoder).
"""

'\nDomain-adapt Layer 2 to phone images WITHOUT retraining RT-DETR.\n \nThe RT-DETR encoder is frozen (you only ever trained the GOA head on cached\nfeatures). So "adapting Layer 2 to phone" = re-cache the encoder\'s 768-d\nfeatures on the HAM+PAD mix, then re-run the SAME GOA head search on the mixed\nfeatures. Encoder weights never move. ~40 min, mostly the GOA swarm.\n \nThis file REPLACES cell G1 of your GOA notebook and adds a per-domain +\nescalated eval. Cells G2, G3, G4 from that notebook run UNCHANGED on the\nXtr/ytr/Xva/yva/Xte/yte this produces. Run P0-P3, then G2-G4, then P5.\n \nLayer 2 is image-only — no metadata here (that lives in Layer 1).\nColab, GPU. Set CKPT_L2 to layer2_rtdetr.pt (the encoder).\n'

In [2]:

# %% ===================================================== CELL P0: frozen encoder
import os, glob, time, json
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import f1_score, accuracy_score

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
import kagglehub

OUT = "/content/drive/MyDrive/amsdds"
CKPT_L2 = f"{OUT}/layer2_rtdetr.pt"           # frozen encoder (backbone+hybrid encoder)
HF = "PekingU/rtdetr_r50vd"
IMG_SIZE, SHORT, BATCH = 512, 588, 16
CACHE = "/content/l2_cache512"                 # 512px cache, HAM+PAD together
SEED = 42; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CACHE, exist_ok=True); torch.manual_seed(SEED); np.random.seed(SEED)

!pip install -q "transformers>=4.46"
from transformers import RTDetrForObjectDetection

class RTDetrClassifier(nn.Module):
    def __init__(self, name, n_classes, drop=0.2):
        super().__init__()
        base = RTDetrForObjectDetection.from_pretrained(name).model
        self.backbone = base.backbone
        self.input_proj = base.encoder_input_proj
        self.encoder = base.encoder
        d, nlev = base.config.d_model, len(base.encoder_input_proj)
        self.head = nn.Sequential(nn.LayerNorm(d * nlev), nn.Dropout(drop),
                                  nn.Linear(d * nlev, n_classes))
    def forward(self, x): raise NotImplementedError  # we only pool

ckL2 = torch.load(CKPT_L2, map_location="cpu", weights_only=False)
CLASSES = ckL2.get("classes", ["akiec","bcc","bkl","df","mel","nv","vasc"])
CI = {c: i for i, c in enumerate(CLASSES)}
enc = RTDetrClassifier(ckL2.get("model_name", HF), len(CLASSES))
enc.load_state_dict(ckL2["state"], strict=False)      # encoder loads; head ignored
enc = enc.eval().to(DEVICE)
for p in enc.parameters(): p.requires_grad_(False)    # FROZEN
print("frozen RT-DETR encoder loaded")

@torch.no_grad()
def pooled(x):
    mask = torch.ones(x.shape[0], x.shape[2], x.shape[3], device=x.device)
    feats = enc.backbone(x, mask)
    srcs = [enc.input_proj[i](f) for i, (f, _) in enumerate(feats)]
    e = enc.encoder(inputs_embeds=srcs)[0]
    return torch.cat([f.mean((2, 3)) for f in e], 1)   # 768-d


Mounted at /content/drive


config.json:   0%|          | 0.00/5.11k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  172MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

frozen RT-DETR encoder loaded


In [3]:

# %% ===================================================== CELL P1: combined df (image-only)
DHAM = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
DPAD = kagglehub.dataset_download("mahdavi1202/skin-cancer")

ham = pd.read_csv(f"{OUT}/splits.csv")
hsrc = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f"{DHAM}/**/*.jpg", recursive=True)}
ham["src"] = ham.image_id.map(hsrc)
ham["uid"] = "ham_" + ham.image_id.astype(str)
ham["path"] = ham.uid.map(lambda u: f"{CACHE}/{u}.jpg")
ham["dataset"] = "ham"
ham = ham[["uid", "src", "path", "y", "split", "dataset"]]

pcsv = next(p for p in glob.glob(f"{DPAD}/**/*.csv", recursive=True))
pad = pd.read_csv(pcsv); pad.columns = [c.strip().lower() for c in pad.columns]
pimg = {os.path.basename(p): p for p in glob.glob(f"{DPAD}/**/*.png", recursive=True)}
CLSMAP = {"ACK":"akiec","BCC":"bcc","SCC":"akiec","MEL":"mel","NEV":"nv","SEK":"bkl"}
pad["dx"] = pad.diagnostic.str.upper().map(CLSMAP)
pad = pad[pad.dx.notna()].copy(); pad["y"] = pad.dx.map(CI)
pad["src"] = pad.img_id.map(pimg)
pad["uid"] = "pad_" + pad.img_id.astype(str).str.replace(".png", "", regex=False)
pad["path"] = pad.uid.map(lambda u: f"{CACHE}/{u}.jpg")
from sklearn.model_selection import StratifiedGroupKFold
fo = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(pad, pad.y, groups=pad.lesion_id))
pad["split"] = "train"
pad.iloc[fo[0][1], pad.columns.get_loc("split")] = "test"
pad.iloc[fo[1][1], pad.columns.get_loc("split")] = "val"
assert pad.groupby("lesion_id").split.nunique().max() == 1
pad["dataset"] = "pad"
pad = pad[["uid", "src", "path", "y", "split", "dataset"]]

df = pd.concat([ham, pad], ignore_index=True)
print("combined:", df.groupby(["dataset", "split"]).size().to_dict())


Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Using Colab cache for faster access to the 'skin-cancer' dataset.
combined: {('ham', 'test'): 1441, ('ham', 'train'): 7116, ('ham', 'val'): 1458, ('pad', 'test'): 445, ('pad', 'train'): 1405, ('pad', 'val'): 448}


In [4]:

# %% ===================================================== CELL P2: cache both at 512 CC
def shades_of_gray(a, power=6):
    a = a.astype(np.float32)
    v = np.power(np.mean(np.power(a, power), (0, 1)), 1.0 / power)
    v = v / (np.sqrt((v ** 2).sum()) + 1e-8)
    return np.clip(a / (v * np.sqrt(3) + 1e-8), 0, 255).astype(np.uint8)
def build(r):
    if os.path.exists(r.path): return
    im = Image.open(r.src).convert("RGB"); w, h = im.size; s = SHORT / min(w, h)
    im = im.resize((round(w*s), round(h*s)), Image.BICUBIC)
    Image.fromarray(shades_of_gray(np.array(im))).save(r.path, quality=95)
todo = df[[not os.path.exists(p) for p in df.path]]
if len(todo):
    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex: list(ex.map(build, list(todo.itertuples())))
    print(f"cached {len(todo)} in {time.time()-t0:.0f}s")
else: print("cache ready")


cached 12313 in 550s


In [5]:

# %% ===================================================== CELL P3: extract features (= GOA G1 output)
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
train_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.65,1.0), ratio=(0.85,1.18)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomApply([T.RandomRotation(30)], p=0.5),
    T.ColorJitter(0.35,0.35,0.25,0.03), T.RandomApply([T.GaussianBlur(5,(0.1,1.5))], p=0.25),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(SHORT), T.CenterCrop(IMG_SIZE), T.ToTensor(), T.Normalize(MEAN, STD)])
class DS(Dataset):
    def __init__(self, frame, tf): self.f = frame.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]; return self.tf(Image.open(r.path).convert("RGB")), int(r.y)

@torch.no_grad()
def extract(frame, tf):
    F_, Y_ = [], []
    for x, y in DataLoader(DS(frame, tf), 32, num_workers=2, pin_memory=True):
        with torch.autocast("cuda", torch.float16, enabled=DEVICE=="cuda"):
            F_.append(pooled(x.to(DEVICE)).float().cpu())
        Y_.append(y)
    return torch.cat(F_).numpy(), torch.cat(Y_).numpy()

tr, va, te = (df[df.split==s] for s in ("train","val","test"))
FEAT = f"{OUT}/rtdetr_feats_hampad.npz"
AUG_COPIES = 2
if os.path.exists(FEAT):
    z = np.load(FEAT, allow_pickle=True)
    Xtr,ytr,Xva,yva,Xte,yte,te_ds,te_uid = (z[k] for k in ("Xtr","ytr","Xva","yva","Xte","yte","te_ds","te_uid"))
    print("loaded cached combined features")
else:
    t0 = time.time()
    Xtr, ytr = extract(tr, eval_tf)
    for k in range(AUG_COPIES):
        Xa, ya = extract(tr, train_tf); Xtr, ytr = np.r_[Xtr, Xa], np.r_[ytr, ya]
        print(f"aug {k+1} ({time.time()-t0:.0f}s)")
    Xva, yva = extract(va, eval_tf)
    Xte, yte = extract(te, eval_tf)
    te_ds, te_uid = te.dataset.values, te.uid.values
    np.savez(FEAT, Xtr=Xtr,ytr=ytr,Xva=Xva,yva=yva,Xte=Xte,yte=yte,te_ds=te_ds,te_uid=te_uid)
    print(f"cached features in {time.time()-t0:.0f}s")
print("train", Xtr.shape, "| val", Xva.shape, "| test", Xte.shape)

# standardise on train (GOA cells expect Ztr/Zva/Zte + the tensors)
mu_f, sd_f = Xtr.mean(0), Xtr.std(0) + 1e-6
Ztr, Zva, Zte = (Xtr-mu_f)/sd_f, (Xva-mu_f)/sd_f, (Xte-mu_f)/sd_f
Ztr_t, ytr_t = torch.tensor(Ztr, device=DEVICE), torch.tensor(ytr, device=DEVICE)
Zva_t, Zte_t = torch.tensor(Zva, device=DEVICE), torch.tensor(Zte, device=DEVICE)
counts = np.bincount(ytr, minlength=len(CLASSES)).astype(np.float32)
print("\n>>> now run GOA cells G2, G3, G4 UNCHANGED. They use Ztr_t/ytr_t/Zva_t/counts.")
print(">>> then run P5 below for per-domain + escalated eval and to save the head.")


aug 1 (558s)
aug 2 (962s)
cached features in 1022s
train (25563, 768) | val (1906, 768) | test (1886, 768)

>>> now run GOA cells G2, G3, G4 UNCHANGED. They use Ztr_t/ytr_t/Zva_t/counts.
>>> then run P5 below for per-domain + escalated eval and to save the head.


In [6]:

# %% ============================================ CELL G2: head trainer (one candidate)
def make_head(hidden, drop, d_in=Ztr.shape[1], n=len(CLASSES)):
    if hidden < 8:                                     # linear head
        return nn.Sequential(nn.Dropout(drop), nn.Linear(d_in, n))
    return nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, n))

def train_head(h, seed=0, return_model=False, verbose=False):
    """h: dict(lr, hidden, drop, wd, cap, smooth, epochs). Returns best val F1
    (and the model + val logits if return_model)."""
    torch.manual_seed(seed)
    net = make_head(int(h["hidden"]), h["drop"]).to(DEVICE)
    w = 1.0 / np.sqrt(counts); w = np.clip(w / w.min(), 1.0, h["cap"])
    lossfn = nn.CrossEntropyLoss(weight=torch.tensor(w, device=DEVICE, dtype=torch.float32),
                                 label_smoothing=h["smooth"])
    opt = torch.optim.AdamW(net.parameters(), lr=h["lr"], weight_decay=h["wd"])
    E = int(h["epochs"]); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
    B, N = 256, len(Ztr_t); best, best_state = -1, None
    for ep in range(E):
        net.train(); perm = torch.randperm(N, device=DEVICE)
        for i in range(0, N, B):
            idx = perm[i:i+B]; opt.zero_grad()
            lossfn(net(Ztr_t[idx]), ytr_t[idx]).backward(); opt.step()
        sched.step()
        net.eval()
        with torch.no_grad(): lv = net(Zva_t).cpu()
        f1 = f1_score(yva, lv.argmax(1).numpy(), average="macro")
        if f1 > best:
            best, best_state = f1, {k: v.clone() for k, v in net.state_dict().items()}
    if return_model:
        net.load_state_dict(best_state); net.eval()
        with torch.no_grad(): lv = net(Zva_t).cpu()
        return best, net, lv
    return best

# baseline: the head you trained end-to-end, re-fit on features (sanity)
t0 = time.time()
base_h = dict(lr=5e-4, hidden=0, drop=0.2, wd=1e-4, cap=2.0, smooth=0.05, epochs=40)
print(f"linear-head baseline val F1 {train_head(base_h):.4f}  ({time.time()-t0:.1f}s per candidate)")


linear-head baseline val F1 0.6723  (7.8s per candidate)


In [7]:

# %% ============================================ CELL G3: grasshopper optimisation
# dims:      log10lr   hidden   drop   log10wd   cap    smooth   epochs
LB = np.array([-4.0,     0,     0.0,   -5.0,     1.0,   0.0,     20])
UB = np.array([-2.0,   1024,    0.6,   -2.0,     5.0,   0.15,    80])
NAMES = ["log10lr", "hidden", "drop", "log10wd", "cap", "smooth", "epochs"]

def decode(x):
    return dict(lr=10 ** x[0], hidden=int(round(x[1] / 64) * 64), drop=float(x[2]),
                wd=10 ** x[3], cap=float(x[4]), smooth=float(x[5]), epochs=int(x[6]))

def fitness(x):                       # minimise
    return -train_head(decode(x))

def goa(n_pop=12, n_iter=12, cmax=1.0, cmin=1e-4, f=0.5, l=1.5, seed=42):
    rng = np.random.RandomState(seed); d = len(LB)
    X = LB + rng.rand(n_pop, d) * (UB - LB)
    # seed one grasshopper with the known-good recipe
    X[0] = np.array([np.log10(5e-4), 0, 0.2, -4, 2.0, 0.05, 40])
    fit = np.array([fitness(x) for x in X])
    best_i = fit.argmin(); target, target_fit = X[best_i].copy(), fit[best_i]
    hist = [(-target_fit, decode(target))]
    print(f"init  best val F1 {-target_fit:.4f}  {decode(target)}")
    S = lambda r: f * np.exp(-r / l) - np.exp(-r)
    for it in range(n_iter):
        c = cmax - it * (cmax - cmin) / n_iter
        Xn = np.zeros_like(X)
        for i in range(n_pop):
            s = np.zeros(d)
            for j in range(n_pop):
                if i == j: continue
                dist = np.linalg.norm(X[j] - X[i]) + 1e-12
                r = 2 + (dist % 2)                          # normalise into [2,4)
                s += c * (UB - LB) / 2 * S(r) * (X[j] - X[i]) / dist
            Xn[i] = np.clip(c * s + target, LB, UB)
        X = Xn
        fit = np.array([fitness(x) for x in X])
        if fit.min() < target_fit:
            target, target_fit = X[fit.argmin()].copy(), fit.min()
        hist.append((-target_fit, decode(target)))
        print(f"iter {it:2d}  c={c:.3f}  swarm best {-fit.min():.4f}  global best {-target_fit:.4f}")
    return target, -target_fit, hist

t0 = time.time()
best_x, best_val, hist = goa(n_pop=12, n_iter=12)
BEST_H = decode(best_x)
print(f"\nGOA done in {(time.time()-t0)/60:.1f} min\nbest val macro-F1 {best_val:.4f}\n{BEST_H}")
json.dump({"best": BEST_H, "val_f1": best_val, "history": [(f, h) for f, h in hist],
           "bounds": {"lb": LB.tolist(), "ub": UB.tolist(), "names": NAMES}},
          open(f"{OUT}/goa_head_search.json", "w"), indent=2)


init  best val F1 0.7027  {'lr': np.float64(0.0002342658105820405), 'hidden': 1024, 'drop': 0.4650796940166687, 'wd': np.float64(0.006584106160121609), 'cap': 4.579309401710596, 'smooth': 0.08968499682166277, 'epochs': 75}
iter  0  c=1.000  swarm best 0.7078  global best 0.7078
iter  1  c=0.917  swarm best 0.7078  global best 0.7078
iter  2  c=0.833  swarm best 0.7078  global best 0.7078
iter  3  c=0.750  swarm best 0.7078  global best 0.7078
iter  4  c=0.667  swarm best 0.7078  global best 0.7078
iter  5  c=0.583  swarm best 0.7078  global best 0.7078
iter  6  c=0.500  swarm best 0.7078  global best 0.7078
iter  7  c=0.417  swarm best 0.7078  global best 0.7078
iter  8  c=0.333  swarm best 0.7078  global best 0.7078
iter  9  c=0.250  swarm best 0.7078  global best 0.7078
iter 10  c=0.167  swarm best 0.7078  global best 0.7078
iter 11  c=0.083  swarm best 0.7078  global best 0.7078

GOA done in 39.5 min
best val macro-F1 0.7078
{'lr': np.float64(0.00023401508370623884), 'hidden': 960, 

In [8]:

# %% ============================================ CELL G4: refit best head, calibrate
# 3 seeds, keep the best val -> guards against a lucky seed in the swarm
cands = [train_head(BEST_H, seed=s, return_model=True) for s in range(3)]
vf1, head, lv = max(cands, key=lambda t: t[0])
print(f"refit val macro-F1 {vf1:.4f} (seeds: {[round(c[0],4) for c in cands]})")

lv_d, yv_d = lv.to(DEVICE), torch.tensor(yva, device=DEVICE)
logT = torch.zeros(1, device=DEVICE, requires_grad=True)
topt = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)
def _cl():
    topt.zero_grad(); L = F.cross_entropy(lv_d / logT.exp(), yv_d); L.backward(); return L
topt.step(_cl); T_GOA = float(logT.exp().detach())
print(f"temperature {T_GOA:.3f}")


refit val macro-F1 0.7078 (seeds: [0.7078, 0.6942, 0.701])
temperature 0.788


In [9]:

# %% ===================================================== CELL P5: eval per-domain + save
# run AFTER G4 (which produced: head, BEST_H, T_GOA, vf1)
with torch.no_grad(): lt = head(Zte_t).cpu()
p_goa = F.softmax(lt / T_GOA, 1).numpy(); pred = p_goa.argmax(1)

for dom in ("ham", "pad"):
    m = te_ds == dom
    print(f"{dom.upper()} test  acc {accuracy_score(yte[m], pred[m]):.4f}  "
          f"macro-F1 {f1_score(yte[m], pred[m], average='macro'):.4f}  (n={m.sum()})")
print(f"COMBINED   acc {accuracy_score(yte, pred):.4f}  macro-F1 {f1_score(yte, pred, average='macro'):.4f}")

# escalated-subset eval — needs the ADAPTED Layer 1's routing on the combined test.
# routing file must have a 'uid' column matching te_uid (ham_<id> / pad_<id>).
ROUTING = f"{OUT}/test_routing_detail_adapted.csv"
if os.path.exists(ROUTING):
    r = pd.read_csv(ROUTING)
    if "uid" in r.columns:
        r = r.set_index("uid").reindex(te_uid).reset_index()
        esc = r["escalated"].fillna(False).values.astype(bool)
        MAL = [CI[c] for c in ("mel","bcc","akiec")]
        pr, yy = pred[esc], yte[esc]; mal = np.isin(yy, MAL)
        print(f"\nescalated {esc.sum()}/{len(esc)}  esc-acc {(pr==yy).mean():.4f}  "
              f"mal-sens {np.isin(pr[mal], MAL).mean():.3f}")
    else:
        print("\nrouting file has no 'uid' column -- regenerate it keyed on uid to score escalated")
else:
    print(f"\n{ROUTING} not found -- adapted Layer 1 routing not ready; per-domain numbers above stand")

torch.save({"head": head.state_dict(), "hparams": BEST_H, "temperature": T_GOA,
            "feat_mu": mu_f, "feat_sd": sd_f, "classes": CLASSES,
            "encoder_ckpt": "layer2_rtdetr.pt", "trained_on": "ham+pad-ufes-20",
            "val_macro_f1": vf1},
           f"{OUT}/layer2_rtdetr_goa_head_hampad.pt")
np.save(f"{OUT}/test_probs_rtdetr_goa_hampad.npy", p_goa)
print("\nsaved layer2_rtdetr_goa_head_hampad.pt (encoder unchanged: layer2_rtdetr.pt)")


HAM test  acc 0.8570  macro-F1 0.7361  (n=1441)
PAD test  acc 0.6382  macro-F1 0.4576  (n=445)
COMBINED   acc 0.8054  macro-F1 0.7230

/content/drive/MyDrive/amsdds/test_routing_detail_adapted.csv not found -- adapted Layer 1 routing not ready; per-domain numbers above stand

saved layer2_rtdetr_goa_head_hampad.pt (encoder unchanged: layer2_rtdetr.pt)


In [10]:
from sklearn.metrics import classification_report, confusion_matrix
m = te_ds == "pad"
print(classification_report(yte[m], pred[m], labels=[CI[c] for c in CLASSES],
      target_names=CLASSES, digits=3, zero_division=0))
print(confusion_matrix(yte[m], pred[m], labels=range(len(CLASSES))))

              precision    recall  f1-score   support

       akiec      0.633     0.720     0.674       175
         bcc      0.703     0.665     0.683       167
         bkl      0.440     0.458     0.449        48
          df      0.000     0.000     0.000         0
         mel      0.600     0.250     0.353        12
          nv      0.688     0.512     0.587        43
        vasc      0.000     0.000     0.000         0

    accuracy                          0.638       445
   macro avg      0.438     0.372     0.392       445
weighted avg      0.643     0.638     0.636       445

[[126  36  11   0   1   1   0]
 [ 53 111   0   1   0   2   0]
 [ 18   3  22   0   0   5   0]
 [  0   0   0   0   0   0   0]
 [  0   0   7   0   3   2   0]
 [  2   8  10   0   1  22   0]
 [  0   0   0   0   0   0   0]]


In [11]:
"""
Patch for the HAM+PAD GOA search: make the head stop ignoring PAD minorities.
Two changes:
  (A) weighted sampling in train_head -> PAD gets 50% of every batch, and within
      PAD, rare classes (melanoma) are oversampled.
  (B) fitness = 0.5*HAM_valF1 + 0.5*PAD_valF1 (present PAD classes only), so the
      swarm optimizes both domains instead of the HAM-dominated combined val.

Run these three cells, THEN re-run G3 (goa), G4 (refit), P5 (eval). Requires
Ztr_t/ytr_t/Zva_t/yva and the train/val frames tr, va from P3 still in memory.
"""

# %% ===================================================== PATCH-A: domain/class sample weights
tr_ds = np.tile(tr.dataset.values, (len(ytr) // len(tr)))      # aug copies repeat tr in order
assert len(tr_ds) == len(ytr), f"tag/len mismatch {len(tr_ds)} vs {len(ytr)}"
va_ds = va.dataset.values

n_ham = (tr_ds == "ham").sum(); n_pad = (tr_ds == "pad").sum()
pad_cls_cnt = np.bincount(ytr[tr_ds == "pad"], minlength=len(CLASSES)).astype(np.float32)
boost = np.where(pad_cls_cnt > 0, 1.0 / np.sqrt(pad_cls_cnt), 0.0)
boost = boost / boost[boost > 0].mean()                        # mean-1 within PAD

SW = np.empty(len(ytr), np.float32)
SW[tr_ds == "ham"] = 0.5 / n_ham                               # HAM half, flat
pad_mask = tr_ds == "pad"
SW[pad_mask] = (0.5 / n_pad) * boost[ytr[pad_mask]]            # PAD half, minority-tilted
SW = torch.tensor(SW / SW.sum(), device=DEVICE)
PAD_PRESENT = sorted(set(yva[va_ds == "pad"].tolist()))
print(f"HAM train {n_ham}, PAD train {n_pad} -> batches now ~50/50")
print("PAD per-class oversample factor:", {CLASSES[c]: round(float(boost[c]), 2) for c in range(len(CLASSES)) if boost[c] > 0})

# %% ===================================================== PATCH-B: weighted train_head
def train_head(h, seed=0, return_model=False, verbose=False):
    torch.manual_seed(seed)
    net = make_head(int(h["hidden"]), h["drop"]).to(DEVICE)
    w = 1.0 / np.sqrt(counts); w = np.clip(w / w.min(), 1.0, h["cap"])
    lossfn = nn.CrossEntropyLoss(weight=torch.tensor(w, device=DEVICE, dtype=torch.float32),
                                 label_smoothing=h["smooth"])
    opt = torch.optim.AdamW(net.parameters(), lr=h["lr"], weight_decay=h["wd"])
    E = int(h["epochs"]); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
    B, N = 256, len(Ztr_t); steps = max(1, N // B)
    best, best_state = -1, None
    for ep in range(E):
        net.train()
        for _ in range(steps):
            idx = torch.multinomial(SW, B, replacement=True)   # weighted, PAD-heavy
            opt.zero_grad(); lossfn(net(Ztr_t[idx]), ytr_t[idx]).backward(); opt.step()
        sched.step(); net.eval()
        with torch.no_grad(): lv = net(Zva_t).cpu()
        pv = lv.argmax(1).numpy()
        hf = f1_score(yva[va_ds == "ham"], pv[va_ds == "ham"], average="macro")
        pf = f1_score(yva[va_ds == "pad"], pv[va_ds == "pad"], average="macro",
                      labels=PAD_PRESENT, zero_division=0)
        score = 0.5 * hf + 0.5 * pf                            # selection metric = balanced
        if score > best:
            best = score
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
    if return_model:
        net.load_state_dict(best_state); net.eval()
        with torch.no_grad(): lv = net(Zva_t).cpu()
        return best, net, lv
    return best

# %% ===================================================== PATCH-C: balanced fitness + baseline
def fitness(x):                                                # minimise
    return -train_head(decode(x))

base = train_head(dict(lr=5e-4, hidden=192, drop=0.05, wd=3e-3, cap=3.8, smooth=0.1, epochs=60),
                  return_model=True)
bv, _, blv = base
bp = blv.argmax(1).numpy()
print(f"balanced baseline: combined-select {bv:.4f} | "
      f"HAM valF1 {f1_score(yva[va_ds=='ham'], bp[va_ds=='ham'], average='macro'):.3f} | "
      f"PAD valF1 {f1_score(yva[va_ds=='pad'], bp[va_ds=='pad'], average='macro', labels=PAD_PRESENT, zero_division=0):.3f}")
print(">>> now re-run G3 (goa), then G4, then P5. Add the PAD melanoma check below to P5.")


/tmp/ipykernel_1218/1471569762.py:20: RuntimeWarning: divide by zero encountered in divide
  boost = np.where(pad_cls_cnt > 0, 1.0 / np.sqrt(pad_cls_cnt), 0.0)


HAM train 21348, PAD train 4215 -> batches now ~50/50
PAD per-class oversample factor: {'akiec': 0.48, 'bcc': 0.5, 'bkl': 0.97, 'mel': 2.15, 'nv': 0.89}
balanced baseline: combined-select 0.6294 | HAM valF1 0.725 | PAD valF1 0.534
>>> now re-run G3 (goa), then G4, then P5. Add the PAD melanoma check below to P5.


In [12]:

# %% ============================================ CELL G3: grasshopper optimisation
# dims:      log10lr   hidden   drop   log10wd   cap    smooth   epochs
LB = np.array([-4.0,     0,     0.0,   -5.0,     1.0,   0.0,     20])
UB = np.array([-2.0,   1024,    0.6,   -2.0,     5.0,   0.15,    80])
NAMES = ["log10lr", "hidden", "drop", "log10wd", "cap", "smooth", "epochs"]

def decode(x):
    return dict(lr=10 ** x[0], hidden=int(round(x[1] / 64) * 64), drop=float(x[2]),
                wd=10 ** x[3], cap=float(x[4]), smooth=float(x[5]), epochs=int(x[6]))

def fitness(x):                       # minimise
    return -train_head(decode(x))

def goa(n_pop=12, n_iter=12, cmax=1.0, cmin=1e-4, f=0.5, l=1.5, seed=42):
    rng = np.random.RandomState(seed); d = len(LB)
    X = LB + rng.rand(n_pop, d) * (UB - LB)
    # seed one grasshopper with the known-good recipe
    X[0] = np.array([np.log10(5e-4), 0, 0.2, -4, 2.0, 0.05, 40])
    fit = np.array([fitness(x) for x in X])
    best_i = fit.argmin(); target, target_fit = X[best_i].copy(), fit[best_i]
    hist = [(-target_fit, decode(target))]
    print(f"init  best val F1 {-target_fit:.4f}  {decode(target)}")
    S = lambda r: f * np.exp(-r / l) - np.exp(-r)
    for it in range(n_iter):
        c = cmax - it * (cmax - cmin) / n_iter
        Xn = np.zeros_like(X)
        for i in range(n_pop):
            s = np.zeros(d)
            for j in range(n_pop):
                if i == j: continue
                dist = np.linalg.norm(X[j] - X[i]) + 1e-12
                r = 2 + (dist % 2)                          # normalise into [2,4)
                s += c * (UB - LB) / 2 * S(r) * (X[j] - X[i]) / dist
            Xn[i] = np.clip(c * s + target, LB, UB)
        X = Xn
        fit = np.array([fitness(x) for x in X])
        if fit.min() < target_fit:
            target, target_fit = X[fit.argmin()].copy(), fit.min()
        hist.append((-target_fit, decode(target)))
        print(f"iter {it:2d}  c={c:.3f}  swarm best {-fit.min():.4f}  global best {-target_fit:.4f}")
    return target, -target_fit, hist

t0 = time.time()
best_x, best_val, hist = goa(n_pop=12, n_iter=12)
BEST_H = decode(best_x)
print(f"\nGOA done in {(time.time()-t0)/60:.1f} min\nbest val macro-F1 {best_val:.4f}\n{BEST_H}")
json.dump({"best": BEST_H, "val_f1": best_val, "history": [(f, h) for f, h in hist],
           "bounds": {"lb": LB.tolist(), "ub": UB.tolist(), "names": NAMES}},
          open(f"{OUT}/goa_head_search.json", "w"), indent=2)


init  best val F1 0.6530  {'lr': np.float64(0.001530485212183147), 'hidden': 64, 'drop': 0.364526911140863, 'wd': np.float64(3.2476735706274485e-05), 'cap': 1.260206371941118, 'smooth': 0.14233283058799998, 'epochs': 77}
iter  0  c=1.000  swarm best 0.6565  global best 0.6565
iter  1  c=0.917  swarm best 0.6569  global best 0.6569
iter  2  c=0.833  swarm best 0.6569  global best 0.6569
iter  3  c=0.750  swarm best 0.6569  global best 0.6569


KeyboardInterrupt: 

In [13]:

# %% ============================================ CELL G4: refit best head, calibrate
# 3 seeds, keep the best val -> guards against a lucky seed in the swarm
cands = [train_head(BEST_H, seed=s, return_model=True) for s in range(3)]
vf1, head, lv = max(cands, key=lambda t: t[0])
print(f"refit val macro-F1 {vf1:.4f} (seeds: {[round(c[0],4) for c in cands]})")

lv_d, yv_d = lv.to(DEVICE), torch.tensor(yva, device=DEVICE)
logT = torch.zeros(1, device=DEVICE, requires_grad=True)
topt = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)
def _cl():
    topt.zero_grad(); L = F.cross_entropy(lv_d / logT.exp(), yv_d); L.backward(); return L
topt.step(_cl); T_GOA = float(logT.exp().detach())
print(f"temperature {T_GOA:.3f}")


refit val macro-F1 0.6438 (seeds: [0.6438, 0.6437, 0.6377])
temperature 0.803


In [14]:

# %% ===================================================== CELL P5: eval per-domain + save
# run AFTER G4 (which produced: head, BEST_H, T_GOA, vf1)
with torch.no_grad(): lt = head(Zte_t).cpu()
p_goa = F.softmax(lt / T_GOA, 1).numpy(); pred = p_goa.argmax(1)

for dom in ("ham", "pad"):
    m = te_ds == dom
    print(f"{dom.upper()} test  acc {accuracy_score(yte[m], pred[m]):.4f}  "
          f"macro-F1 {f1_score(yte[m], pred[m], average='macro'):.4f}  (n={m.sum()})")
print(f"COMBINED   acc {accuracy_score(yte, pred):.4f}  macro-F1 {f1_score(yte, pred, average='macro'):.4f}")

# escalated-subset eval — needs the ADAPTED Layer 1's routing on the combined test.
# routing file must have a 'uid' column matching te_uid (ham_<id> / pad_<id>).
ROUTING = f"{OUT}/test_routing_detail_adapted.csv"
if os.path.exists(ROUTING):
    r = pd.read_csv(ROUTING)
    if "uid" in r.columns:
        r = r.set_index("uid").reindex(te_uid).reset_index()
        esc = r["escalated"].fillna(False).values.astype(bool)
        MAL = [CI[c] for c in ("mel","bcc","akiec")]
        pr, yy = pred[esc], yte[esc]; mal = np.isin(yy, MAL)
        print(f"\nescalated {esc.sum()}/{len(esc)}  esc-acc {(pr==yy).mean():.4f}  "
              f"mal-sens {np.isin(pr[mal], MAL).mean():.3f}")
    else:
        print("\nrouting file has no 'uid' column -- regenerate it keyed on uid to score escalated")
else:
    print(f"\n{ROUTING} not found -- adapted Layer 1 routing not ready; per-domain numbers above stand")

torch.save({"head": head.state_dict(), "hparams": BEST_H, "temperature": T_GOA,
            "feat_mu": mu_f, "feat_sd": sd_f, "classes": CLASSES,
            "encoder_ckpt": "layer2_rtdetr.pt", "trained_on": "ham+pad-ufes-20",
            "val_macro_f1": vf1},
           f"{OUT}/layer2_rtdetr_goa_head_hampad.pt")
np.save(f"{OUT}/test_probs_rtdetr_goa_hampad.npy", p_goa)
print("\nsaved layer2_rtdetr_goa_head_stuff_tried.pt (encoder unchanged: layer2_rtdetr.pt)")


HAM test  acc 0.8536  macro-F1 0.7203  (n=1441)
PAD test  acc 0.6225  macro-F1 0.5452  (n=445)
COMBINED   acc 0.7990  macro-F1 0.7082

/content/drive/MyDrive/amsdds/test_routing_detail_adapted.csv not found -- adapted Layer 1 routing not ready; per-domain numbers above stand

saved layer2_rtdetr_goa_head_stuff_tried.pt (encoder unchanged: layer2_rtdetr.pt)


In [15]:
from sklearn.metrics import classification_report, confusion_matrix
m = te_ds == "pad"
print(classification_report(yte[m], pred[m], labels=[CI[c] for c in CLASSES],
      target_names=CLASSES, digits=3, zero_division=0))
print(confusion_matrix(yte[m], pred[m], labels=range(len(CLASSES))))

              precision    recall  f1-score   support

       akiec      0.615     0.703     0.656       175
         bcc      0.680     0.623     0.650       167
         bkl      0.420     0.438     0.429        48
          df      0.000     0.000     0.000         0
         mel      0.500     0.250     0.333        12
          nv      0.722     0.605     0.658        43
        vasc      0.000     0.000     0.000         0

    accuracy                          0.622       445
   macro avg      0.420     0.374     0.389       445
weighted avg      0.626     0.622     0.621       445

[[123  38  13   0   1   0   0]
 [ 58 104   2   0   1   2   0]
 [ 17   5  21   0   0   5   0]
 [  0   0   0   0   0   0   0]
 [  0   0   6   0   3   3   0]
 [  2   6   8   0   1  26   0]
 [  0   0   0   0   0   0   0]]


In [16]:
m = te_ds == "pad"
mel_mask = (yte[m] == CI["mel"])
mal_prob = p_goa[m][:, [CI["mel"], CI["bcc"], CI["akiec"]]].sum(1)
print("malignant-prob on PAD melanomas:", np.round(np.sort(mal_prob[mel_mask])[::-1], 3))

malignant-prob on PAD melanomas: [0.81  0.721 0.543 0.391 0.36  0.302 0.259 0.191 0.169 0.109 0.005 0.004]


In [18]:
# --- val-tuned malignant risk threshold ---
from sklearn.metrics import f1_score  # already imported

# 1. head probabilities on PAD val
with torch.no_grad():
    p_va = F.softmax(head(Zva_t) / T_GOA, 1).cpu().numpy()

MAL = [CI["mel"], CI["bcc"], CI["akiec"]]
vmask = va_ds == "pad"
mal_prob_va = p_va[vmask][:, MAL].sum(1)
mal_true_va = np.isin(yva[vmask], MAL)

# 2. sweep on VAL, pick the knee: highest specificity that keeps sensitivity >= target
TARGET_SENS = 0.96
best_thr, best_spec = 0.30, -1
print("VAL sweep:")
for thr in np.round(np.arange(0.10, 0.61, 0.02), 2):
    flag = mal_prob_va >= thr
    sens = flag[mal_true_va].mean()
    spec = (~flag[~mal_true_va]).mean()
    print(f"  thr {thr:.2f}: sens {sens:.3f}  spec {spec:.3f}")
    if sens >= TARGET_SENS and spec > best_spec:
        best_thr, best_spec = float(thr), spec
print(f"\nchosen on val: risk_threshold = {best_thr:.2f}  (val spec {best_spec:.3f})")

# 3. report — do NOT pick — on PAD test
tmask = te_ds == "pad"
mal_prob_te = p_goa[tmask][:, MAL].sum(1)
mal_true_te = np.isin(yte[tmask], MAL)
flag = mal_prob_te >= best_thr
print(f"PAD TEST at {best_thr:.2f}: sensitivity {flag[mal_true_te].mean():.3f}  "
      f"specificity {(~flag[~mal_true_te]).mean():.3f}")

VAL sweep:
  thr 0.10: sens 1.000  spec 0.233
  thr 0.12: sens 0.994  spec 0.244
  thr 0.14: sens 0.986  spec 0.256
  thr 0.16: sens 0.983  spec 0.302
  thr 0.18: sens 0.983  spec 0.314
  thr 0.20: sens 0.978  spec 0.372
  thr 0.22: sens 0.970  spec 0.395
  thr 0.24: sens 0.964  spec 0.442
  thr 0.26: sens 0.964  spec 0.453
  thr 0.28: sens 0.959  spec 0.453
  thr 0.30: sens 0.959  spec 0.488
  thr 0.32: sens 0.956  spec 0.512
  thr 0.34: sens 0.956  spec 0.523
  thr 0.36: sens 0.956  spec 0.547
  thr 0.38: sens 0.950  spec 0.581
  thr 0.40: sens 0.948  spec 0.593
  thr 0.42: sens 0.945  spec 0.628
  thr 0.44: sens 0.942  spec 0.674
  thr 0.46: sens 0.934  spec 0.709
  thr 0.48: sens 0.928  spec 0.721
  thr 0.50: sens 0.928  spec 0.733
  thr 0.52: sens 0.925  spec 0.767
  thr 0.54: sens 0.909  spec 0.767
  thr 0.56: sens 0.901  spec 0.767
  thr 0.58: sens 0.892  spec 0.779
  thr 0.60: sens 0.892  spec 0.802

chosen on val: risk_threshold = 0.26  (val spec 0.453)
PAD TEST at 0.26: sensi

In [19]:
import os, glob, json
D = "/content/drive/MyDrive/amsdds"

print("=== FILES ===")
for p in sorted(glob.glob(f"{D}/**/*", recursive=True)):
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p.replace(D+'/','')}")

print("\n=== CONFIGS / METRICS (contents) ===")
for p in sorted(glob.glob(f"{D}/**/*.yaml", recursive=True) +
                glob.glob(f"{D}/**/*.json", recursive=True)):
    print(f"\n----- {p.replace(D+'/','')}")
    print(open(p).read()[:1500])

print("\n=== CSV headers + row counts ===")
import pandas as pd
for p in sorted(glob.glob(f"{D}/**/*.csv", recursive=True)):
    try:
        df = pd.read_csv(p)
        print(f"\n----- {p.replace(D+'/','')}  ({len(df)} rows)")
        print("cols:", list(df.columns))
    except Exception as e:
        print(f"{p}: {e}")

print("\n=== .pt checkpoint keys ===")
import torch
for p in sorted(glob.glob(f"{D}/**/*.pt", recursive=True)):
    try:
        c = torch.load(p, map_location="cpu", weights_only=False)
        keys = list(c.keys()) if isinstance(c, dict) else "raw state_dict"
        meta = {k: v for k, v in c.items() if k not in ("state","model_state","state_dict","head")} if isinstance(c, dict) else {}
        # trim big/verbose values
        meta = {k: (v if len(str(v)) < 120 else f"<{type(v).__name__}>") for k, v in meta.items()}
        print(f"\n----- {p.replace(D+'/','')}\n  keys: {keys}\n  meta: {meta}")
    except Exception as e:
        print(f"{p}: {e}")

=== FILES ===
   17.04 MB  baseline_mobilenetv3.pt
    0.00 MB  configs/thresholds.yaml
   69.09 MB  dinov2_feats.npz
    0.00 MB  goa_head_search.json
    0.00 MB  layer1_comparison.csv
    1.61 MB  layer1_dinov2_head.pt
    1.61 MB  layer1_dinov2_v1.pt
   17.04 MB  layer1_mobilenetv3_best.pt
   17.72 MB  layer1_multimodal_mobilenetv3.pt
  142.42 MB  layer2_rtdetr.pt
  142.43 MB  layer2_rtdetr_ckpt_tmp.pt
    0.61 MB  layer2_rtdetr_goa_head.pt
    2.99 MB  layer2_rtdetr_goa_head_hampad.pt
    0.37 MB  layer2_train_subset.csv
    0.00 MB  metrics_mobilenetv3.json
    0.00 MB  metrics_rtdetr.json
    4.78 MB  ood_scores.npz
   13.18 MB  ood_scores_mobilenet.npz
    0.00 MB  routing_results.json
   74.68 MB  rtdetr_feats.npz
   90.46 MB  rtdetr_feats_hampad.npz
    1.45 MB  splits.csv
    0.04 MB  test_probs_rtdetr.npy
    0.04 MB  test_probs_rtdetr_goa.npy
    0.05 MB  test_probs_rtdetr_goa_hampad.npy
    0.09 MB  test_routing_detail.csv
    0.01 MB  threshold_sweep.csv
    0.12 MB  thr